# Web App Back End Tests

Tests for the back end of the Structure Relations web application.
Uses *RS.GJS_Struct_Tests.BRBL BH.dcm* file located in the *test_data* folder.

## 1. Import Required Libraries

First, let's import the necessary libraries including our custom DicomStructureFile class.

In [1]:
# Import required libraries
from typing import List, Tuple
#import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import shapely

# Add the src directory to the Python path
#sys.path.append('src')

# Import our custom DicomStructureFile class
from dicom import DicomStructureFile

# Import related classes
from types_and_classes import SliceIndexType
from contours import ContourPoints
from contour_plotting import plot_roi_slice

from structure_set import StructureSet
from relations import RelationshipType

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
%matplotlib inline

In [3]:
# Define the path to the test DICOM file
tests_dir = Path.cwd() / r'Tests'
tests_dir = tests_dir.resolve()
test_file_name = 'RS.GJS_Struct_Tests.BRBL BH.dcm'

print(f"=== Loading DICOM file ===: {test_file_name}")
dicom_file = DicomStructureFile(
    top_dir=tests_dir,
    file_name=test_file_name
)
filtered_contours = dicom_file.filter_exclusions()

structure_set = StructureSet(dicom_structure_file=dicom_file)

INFO:dicom:Successfully loaded DICOM dataset from RS.GJS_Struct_Tests.BRBL BH.dcm


=== Loading DICOM file ===: RS.GJS_Struct_Tests.BRBL BH.dcm


INFO:dicom:Extracted 626 contours from 10 ROIs
INFO:dicom:Found 0 frame-of-reference matches and 0 other matches for structure set RS.GJS_Struct_Tests.BRBL BH.dcm
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:dicom:Calculated resolution from structure 'BODY': 0.1 cm/pixel
INFO:dicom:Filtered 0 contours from 1 excluded ROIs. Remaining: 626 contours from 9 ROIs
INFO:structure_set:Building StructureSet from 626 contour points
INFO:structure_set:Calculating relationships for 9 structures
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: invalid value encountered in relate
  return lib.relate(a, b, **kwargs)
d:\.conda\envs\StructureRelations\Lib\site-packages\shapely\predicates.py:1171: RuntimeWarning: divide by zero encountered in relate
  return lib.

In [4]:
structure_set.summary()

,ROI,Name,Physical_Volume,Exterior_Volume,Hull_Volume,Num_Contours,Num_Regions,Num_Slices,Slice_Range,StructureName,DICOM_Type,Code,CodeScheme,CodeMeaning
0,9,BODY,43449.75,43449.75,46139.28,142,1,112,-13.75 to 14.00,BODY,EXTERNAL,BODY,99VMS_STRUCTCODE,Body
1,11,Cavity,75.00,75.00,83.03,19,1,17,-0.50 to 3.50,Cavity,GTV,GTVp,99VMS_STRUCTCODE,Primary Gross Tumor Volume
2,12,CTV Cavity,276.82,276.82,289.66,29,1,25,-1.50 to 4.50,CTV Cavity,CTV,CTV_High,99VMS_STRUCTCODE,Clinical Target Volume High Risk
3,13,eval PTV,432.80,432.80,447.07,36,1,29,-2.00 to 5.00,eval PTV,,PTVp,99VMS_STRUCTCODE,Primary Planning Target Volume
4,14,PTV Cavity,454.35,454.35,468.37,36,1,29,-2.00 to 5.00,PTV Cavity,,PTV_High,99VMS_STRUCTCODE,Planning Target Volume High Risk
5,17,Heart,805.99,805.99,810.46,41,1,35,-10.50 to -2.00,Heart,,7088,FMA,Heart
6,18,Lung B,5189.28,5189.28,6236.53,208,2,89,-13.75 to 8.25,Lung B,,68877,FMA,Pair of lungs
7,19,Lung L,2460.92,2460.92,3068.97,105,1,89,-13.75 to 8.25,Lung L,,7310,FMA,Left lung
8,20,Lung R,2818.39,2818.39,3282.93,107,1,86,-13.75 to 7.50,Lung R,,7309,FMA,Right lung


In [12]:
row_rois = col_rois = [11, 12, 13, 14, 17, 18, 19, 20]
matrix = structure_set.get_relationship_matrix(row_rois, col_rois,
                                               use_symbols=False)
print(matrix)

Structure_B    Cavity CTV Cavity  eval PTV PTV Cavity     Heart     Lung B  \
Structure_A                                                                  
Cavity         Equals   Overlaps  Overlaps   Overlaps  Disjoint   Disjoint   
CTV Cavity   Overlaps     Equals  Overlaps   Overlaps  Disjoint   Disjoint   
eval PTV     Overlaps   Overlaps    Equals   Overlaps  Disjoint   Disjoint   
PTV Cavity   Overlaps   Overlaps  Overlaps     Equals  Disjoint   Disjoint   
Heart        Disjoint   Disjoint  Disjoint   Disjoint    Equals    Borders   
Lung B       Disjoint   Disjoint  Disjoint   Disjoint   Borders     Equals   
Lung L       Disjoint   Disjoint  Disjoint   Disjoint   Borders   Overlaps   
Lung R       Disjoint   Disjoint  Disjoint   Disjoint   Borders  Partition   

Structure_B    Lung L    Lung R  
Structure_A                      
Cavity       Disjoint  Disjoint  
CTV Cavity   Disjoint  Disjoint  
eval PTV     Disjoint  Disjoint  
PTV Cavity   Disjoint  Disjoint  
Heart         B